# 06_icf_prediction  (oesophagus)

The recovery analysis for the ICF strand. It links the physiotherapy conclusion and the preoperative ICF classifier scores to the surgical outcomes, and asks three questions:

* RQ1: does the preoperative physiotherapy conclusion, low or high, predict pulmonary complications, and does it add to what the surgical preoperative model already knows
* RQ2: do the preoperative ICF classifier scores predict complications, and do they add to the surgical preoperative model
* Validation: does the ICF signal, exercise tolerance in particular, line up with the physiotherapy conclusion

Pulmonary complications are the primary outcome, because the conclusion is a pulmonary risk judgement, with major complications and prolonged stay as secondaries.

Decisions. Only notes dated before surgery are used, so nothing postoperative leaks in. Features are complete case, no imputation. The conclusion is the patient's single preoperative one. ICF qualifiers run from 0, no problem, upward, so a higher score is worse function.

## 1. Setup

In [ ]:
import os
for v in ['OMP_NUM_THREADS','OPENBLAS_NUM_THREADS','MKL_NUM_THREADS','NUMEXPR_NUM_THREADS']: os.environ[v]='1'
import pandas as pd, numpy as np, warnings
from pathlib import Path
warnings.filterwarnings('ignore')
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
from scipy import stats as st
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import roc_auc_score
np.random.seed(42)
THESIS=Path(r"Z:\Users\predicting recovery\AmanDeep\Thesis")
DERIVED=THESIS/"data_derived"; ICF_OUT=THESIS/"outputs"/"icf"; ICF_OUT.mkdir(parents=True, exist_ok=True)
coh=pd.read_csv(DERIVED/"cohort_clean.csv")
notes=pd.read_csv(ICF_OUT/"icf_notes_with_conclusion.csv")
def pad7(s): return pd.Series(s).astype(str).str.strip().str.replace(r'\.0$','',regex=True).str.zfill(7)
coh['MDN']=pad7(coh['upn']); notes['MDN']=pad7(notes['MDN'])
print('cohort', coh.shape, '| notes', notes.shape)

## 2. Keep oesophagus patients and the preoperative notes only

In [ ]:
# Eight-week preoperative window, anchored on the assessment (Edwin's decision, 15 June 2026).
# Set PREOP_WINDOW_DAYS=None to use all preoperative notes as a sensitivity check.
PREOP_WINDOW_DAYS=56
coh['surgery_dt']=pd.to_datetime(coh['surgery_date'], errors='coerce')
notes['note_dt']=pd.to_datetime(notes['NOTITIE_DATUM_dt'], errors='coerce')
sd=coh.set_index('MDN')['surgery_dt']
n=notes[notes['MDN'].isin(set(coh['MDN']))].copy()
n['surgery_dt']=n['MDN'].map(sd)
preop=n[n['note_dt'].notna() & n['surgery_dt'].notna() & (n['note_dt'] < n['surgery_dt'])].copy()
if PREOP_WINDOW_DAYS is not None:
    preop=preop[preop['note_dt'] >= preop['surgery_dt'] - pd.Timedelta(days=PREOP_WINDOW_DAYS)].copy()
icf_cols=[c for c in notes.columns if c.endswith('_lvl')]
print('preoperative window:', 'all preoperative notes' if PREOP_WINDOW_DAYS is None else f'{PREOP_WINDOW_DAYS} days before surgery')
print(f"oesophagus patients: {coh['MDN'].nunique()}")
print(f"  with any ICF notes:        {n['MDN'].nunique()}")
print(f"  with preoperative notes:   {preop['MDN'].nunique()}")
print(f"preoperative notes: {len(preop):,}")

## 3. Preoperative ICF features per patient

For each clinically relevant domain we take the last preoperative value, the note closest to surgery, and the worst preoperative value. Exercise tolerance is the domain Edwin singles out.

In [ ]:
KEY=['INS-B455_lvl','ADM-B440_lvl','MBW-B530_lvl','ETN-D550_lvl','ENR-B1300_lvl','FAC-D540_lvl']
KEY=[c for c in KEY if c in icf_cols]
preop=preop.sort_values('note_dt')
feat=pd.DataFrame({'MDN':sorted(preop['MDN'].unique())}).set_index('MDN')
for c in KEY:
    g=preop.dropna(subset=[c]).groupby('MDN')[c]
    feat[c.split('-')[0]+'_last']=g.last(); feat[c.split('-')[0]+'_max']=g.max()
feat['icf_n_preop_notes']=preop.groupby('MDN').size()
feat=feat.reset_index()
cov=pd.DataFrame({'feature':[c for c in feat.columns if c!='MDN'],
                  'pct_present':[round(100*feat[c].notna().mean(),1) for c in feat.columns if c!='MDN']})
print("Preoperative ICF feature coverage (within patients who have preop notes):")
print(cov.to_string(index=False))

## 4. Conclusion per patient and the analysis table

In [ ]:
concl=preop.dropna(subset=['conc_binary']).groupby('MDN')['conc_binary'].first().rename('conclusion').reset_index()
# fall back to any note conclusion if a patient has no preop note carrying it
allc=notes.dropna(subset=['conc_binary']).groupby('MDN')['conc_binary'].first().rename('conclusion_any').reset_index()
dat=coh.merge(feat, on='MDN', how='left').merge(concl, on='MDN', how='left').merge(allc, on='MDN', how='left')
dat['conclusion']=dat['conclusion'].fillna(dat['conclusion_any'])
dat['conclusion_high']=dat['conclusion'].map({'high':1,'low':0})
print(f"oesophagus with a conclusion: {dat['conclusion'].notna().sum()}  "
      f"(high {int((dat['conclusion']=='high').sum())}, low {int((dat['conclusion']=='low').sum())})")
print(f"oesophagus with exercise-tolerance preop score (INS_last): {dat['INS_last'].notna().sum()}")
print(f"oesophagus with any preop ICF feature: {dat[[c for c in dat.columns if c.endswith('_last')]].notna().any(axis=1).sum()}")

## 5. Functions (lean, calibrated, statsmodels free)

In [ ]:
CONT_BASE=['age_at_surgery','bmi','charlson_cci','ct_stage','cn_stage']
def pipe(cols,C=1.0,penalty='l2'):
    cont=[c for c in cols if c in CONT_BASE or c.endswith('_last') or c.endswith('_max') or c=='icf_n_preop_notes']
    pre=ColumnTransformer([('s',StandardScaler(),cont)],remainder='passthrough')
    return Pipeline([('t',pre),('lr',LogisticRegression(penalty=penalty,C=C,solver='liblinear',max_iter=5000,random_state=42))])
def bestC(X,y):
    cont=[c for c in X.columns if c in CONT_BASE or c.endswith('_last') or c.endswith('_max') or c=='icf_n_preop_notes']
    pre=ColumnTransformer([('s',StandardScaler(),cont)],remainder='passthrough')
    m=LogisticRegressionCV(Cs=10,cv=5,scoring='neg_log_loss',penalty='l2',solver='liblinear',max_iter=5000,random_state=42)
    Pipeline([('t',pre),('lr',m)]).fit(X,y); return float(m.C_[0])
def auc_ci(X,y,nb=300,rs=42):
    y=np.asarray(y); C=bestC(X,y)
    yp=cross_val_predict(pipe(list(X.columns),C),X,y,cv=StratifiedKFold(5,shuffle=True,random_state=rs),method='predict_proba')[:,1]
    rng=np.random.RandomState(rs); b=[roc_auc_score(y[i],yp[i]) for i in (rng.choice(len(y),len(y),True) for _ in range(nb)) if len(np.unique(y[i]))>1]
    return roc_auc_score(y,yp), np.percentile(b,2.5), np.percentile(b,97.5), yp
SURG=['age_at_surgery','sex_male','bmi','asascore','comlong','charlson_cci','anamok_prior_surgery',
      'immunosuppression','neoadj_received','neoadj_chemoradiation','ct_stage','cn_stage','histology_adeno']
SURG=[c for c in SURG if c in dat.columns]
print('surgical preop predictors:', len(SURG))

## 6. RQ1: does the physiotherapy conclusion predict and add value

First the plain association, the complication rate by conclusion, then the model comparison: ASA alone, the conclusion alone, the surgical preoperative model, and the surgical model plus the conclusion, on the same patients.

In [ ]:
rq1_rows=[]
for o in ['pulmonary','cd_ge_IIIb','los_prolonged']:
    d=dat.dropna(subset=['conclusion_high',o]+SURG).copy(); y=d[o].astype(int)
    if y.nunique()<2 or len(d)<60: continue
    rate=d.groupby('conclusion')[o].mean()
    a_asa,_,_,_=auc_ci(d[['asascore']],y); a_con,_,_,_=auc_ci(d[['conclusion_high']],y)
    a_su,_,_,_=auc_ci(d[SURG],y); a_sc,lo,hi,_=auc_ci(d[SURG+['conclusion_high']],y)
    # paired bootstrap delta (surgical+conclusion vs surgical)
    _,_,_,p1=auc_ci(d[SURG],y); _,_,_,p2=auc_ci(d[SURG+['conclusion_high']],y); yv=y.values
    rng=np.random.RandomState(1); dd=[roc_auc_score(yv[i],p2[i])-roc_auc_score(yv[i],p1[i]) for i in (rng.choice(len(yv),len(yv),True) for _ in range(300)) if len(np.unique(yv[i]))>1]
    rq1_rows.append({'outcome':o,'N':len(d),'events':int(y.sum()),
                     'rate_low':round(rate.get('low',np.nan),3),'rate_high':round(rate.get('high',np.nan),3),
                     'ASA':round(a_asa,3),'conclusion_alone':round(a_con,3),'surgical':round(a_su,3),
                     'surgical+conclusion':round(a_sc,3),'gain':round(a_sc-a_su,3),
                     'gain_ci':f"[{np.percentile(dd,2.5):.3f},{np.percentile(dd,97.5):.3f}]"})
rq1=pd.DataFrame(rq1_rows); rq1.to_csv(ICF_OUT/'rq1_conclusion.csv',index=False)
print(rq1.to_string(index=False))

## 7. RQ2: do the preoperative ICF scores predict and add value

Each ICF domain is added to the surgical model one at a time, on its own complete-case set, to see which domain carries signal. Then the well covered domains are combined.

In [ ]:
rq2_rows=[]
icf_last=[c for c in dat.columns if c.endswith('_last')]
for o in ['pulmonary','cd_ge_IIIb','los_prolonged']:
    for fcol in icf_last:
        d=dat.dropna(subset=[fcol,o]+SURG).copy(); y=d[o].astype(int)
        if y.nunique()<2 or len(d)<60: continue
        a_su,_,_,_=auc_ci(d[SURG],y); a_si,_,_,_=auc_ci(d[SURG+[fcol]],y); a_al,_,_,_=auc_ci(d[[fcol]],y)
        rq2_rows.append({'outcome':o,'icf_feature':fcol,'N':len(d),'events':int(y.sum()),
                         'icf_alone':round(a_al,3),'surgical':round(a_su,3),'surgical+icf':round(a_si,3),'gain':round(a_si-a_su,3)})
rq2=pd.DataFrame(rq2_rows); rq2.to_csv(ICF_OUT/'rq2_icf_features.csv',index=False)
print(rq2.to_string(index=False))

## 8. Validation: does exercise tolerance line up with the conclusion

In [ ]:
val=dat.dropna(subset=['conclusion']).copy()
if 'INS_max' in val.columns and val['INS_max'].notna().any():
    by=val.dropna(subset=['INS_max']).groupby('conclusion')['INS_max'].agg(['size','mean','median'])
    print("Worst preoperative exercise-tolerance score (INS) by conclusion:"); print(by.round(2).to_string())
    g_hi=val.loc[val['conclusion']=='high','INS_max'].dropna(); g_lo=val.loc[val['conclusion']=='low','INS_max'].dropna()
    if len(g_hi)>=5 and len(g_lo)>=5:
        print(f"\nMann-Whitney high vs low: p={st.mannwhitneyu(g_hi,g_lo).pvalue:.4f}")
print(f"\nValidation patients with both a conclusion and an INS score: {val['INS_max'].notna().sum()}")

## 9. Figures

In [ ]:
# RQ1: complication rate by conclusion
if len(rq1):
    fig,ax=plt.subplots(figsize=(8,4)); x=np.arange(len(rq1)); w=0.35
    ax.bar(x-w/2,rq1['rate_low'],w,label='low'); ax.bar(x+w/2,rq1['rate_high'],w,label='high')
    ax.set_xticks(x); ax.set_xticklabels(rq1['outcome']); ax.set_ylabel('complication rate'); ax.legend(); ax.set_title('Outcome rate by physiotherapy conclusion')
    plt.tight_layout(); plt.savefig(ICF_OUT/'rq1_rate_by_conclusion.png',dpi=120); plt.close(fig)
# RQ2: added value of ICF over surgical, pulmonary
sub=rq2[rq2['outcome']=='pulmonary']
if len(sub):
    fig,ax=plt.subplots(figsize=(9,4)); x=np.arange(len(sub))
    ax.bar(x-0.2,sub['surgical'],0.4,label='surgical'); ax.bar(x+0.2,sub['surgical+icf'],0.4,label='surgical + ICF')
    ax.axhline(0.5,ls=':',color='grey'); ax.set_xticks(x); ax.set_xticklabels(sub['icf_feature'],rotation=30,ha='right')
    ax.set_ylim(0.4,0.75); ax.set_ylabel('cross validated AUC'); ax.legend(); ax.set_title('Pulmonary: ICF added to the surgical model')
    plt.tight_layout(); plt.savefig(ICF_OUT/'rq2_pulmonary_added_value.png',dpi=120); plt.close(fig)
print('figures saved to', ICF_OUT)

## 10. Reading the results

* RQ1 tells us whether Edwin's conclusion separates the outcomes and whether it adds to the surgical model. The rate by conclusion is the plain clinical signal; the gain column is what it adds on top of the surgical preoperative variables, on the same patients.
* RQ2 tells us, domain by domain, whether the preoperative ICF scores add to the surgical model, with exercise tolerance the one to watch.
* The validation checks that the objective ICF exercise tolerance score is worse in the patients the physiotherapist judged high risk, which would support both signals measuring the same underlying fitness.
* As with the surgical work, the honest expectation is modest discrimination; the value is a calibrated, interpretable read on whether the assessment and the ICF scores carry information beyond the surgical record.

In [ ]:
# Full preoperative ICF profile across all domains (requested by Marike), split by physiotherapy conclusion.
# Higher exercise tolerance is better functioning (confirmed by the clinical team); other domains follow the classifier coding.
ICF_NAMES={'ENR-B1300_lvl':'Energy and drive','ATT-B140_lvl':'Attention','STM-B152_lvl':'Mood',
  'ADM-B440_lvl':'Respiration','INS-B455_lvl':'Exercise tolerance','MBW-B530_lvl':'Weight maintenance',
  'FAC-D540_lvl':'Walking','ETN-D550_lvl':'Eating','BER-D840-D859_lvl':'Work and employment',
  'SOP-B280_lvl':'Pain','SLP-B134_lvl':'Sleep','FML-D760_lvl':'Family relationships',
  'HLC-B164_lvl':'Higher level cognition','MAE-D465_lvl':'Moving around'}
ppv=preop.sort_values('note_dt').groupby('MDN')[icf_cols].last()
ppv=ppv.merge(dat.set_index('MDN')['conclusion'], left_index=True, right_index=True, how='left')
rows=[]
for c in icf_cols:
    if ppv[c].notna().sum()<30: continue
    rows.append({'domain':ICF_NAMES.get(c,c.split('-')[0]),'n':int(ppv[c].notna().sum()),
                 'mean_low':ppv.loc[ppv['conclusion']=='low',c].mean(),
                 'mean_high':ppv.loc[ppv['conclusion']=='high',c].mean(),
                 'mean_all':ppv[c].mean()})
prof_df=pd.DataFrame(rows).dropna(subset=['mean_low','mean_high']).sort_values('mean_all')
prof_df.to_csv(ICF_OUT/'icf_profile_preoperative.csv', index=False)
y=np.arange(len(prof_df)); h=0.38
fig,ax=plt.subplots(figsize=(8,5.5))
ax.barh(y-h/2, prof_df['mean_low'], h, label='Low conclusion', color='#1f77b4')
ax.barh(y+h/2, prof_df['mean_high'], h, label='High conclusion', color='#d62728')
ax.set_yticks(y); ax.set_yticklabels(prof_df['domain'])
ax.set_xlabel('Mean preoperative ICF level'); ax.set_title('Preoperative ICF profile by physiotherapy conclusion')
ax.legend(frameon=False); plt.tight_layout()
plt.savefig(ICF_OUT/'icf_profile_preoperative.png', dpi=300); plt.close(fig)
print('saved icf_profile_preoperative.png and .csv to', ICF_OUT)
print(prof_df.to_string(index=False))


In [ ]:
# ICF categories as descriptive variables (Marike's request): characteristics table per domain.
ICF_NAMES_D={'ENR-B1300_lvl':'Energy and drive','ATT-B140_lvl':'Attention','STM-B152_lvl':'Mood',
  'ADM-B440_lvl':'Respiration','INS-B455_lvl':'Exercise tolerance','MBW-B530_lvl':'Weight maintenance',
  'FAC-D540_lvl':'Walking','ETN-D550_lvl':'Eating','BER-D840-D859_lvl':'Work and employment',
  'SOP-B280_lvl':'Pain','SLP-B134_lvl':'Sleep','FML-D760_lvl':'Family relationships',
  'HLC-B164_lvl':'Higher level cognition','MAE-D465_lvl':'Moving around'}
desc_rows=[]
for c in icf_cols:
    v=ppv[c].dropna()
    if len(v)<30: continue
    desc_rows.append({'ICF category':ICF_NAMES_D.get(c,c.split('-')[0]),
                      'n with a preoperative value':int(len(v)),
                      'mean':round(float(v.mean()),2),'SD':round(float(v.std()),2),
                      'median':round(float(v.median()),1),
                      'IQR':f"{round(float(v.quantile(.25)),1)} to {round(float(v.quantile(.75)),1)}"})
icf_desc=pd.DataFrame(desc_rows).sort_values('n with a preoperative value',ascending=False)
icf_desc.to_csv(ICF_OUT/'icf_categories_descriptive.csv',index=False)
print(icf_desc.to_string(index=False))
